# CE-4 Project: Online Consultation & Prescription Management Dashboard

**Registration Number:** 25SSDS415051  
**Course:** Data Science and Analytics  
**Technology:** Python, Tkinter, SQLite

## Project objective
Build a desktop database application for a telemedicine platform. It stores doctors, consultation sessions, and prescription items in three related SQLite tables.

### Features included
- Three related tables with foreign keys and `ON DELETE CASCADE`.
- Automatic sample data when the database is empty.
- Add a consultation and its prescription in one operation.
- Reuse an existing doctor profile or create a new one.
- Delete a doctor, consultation, or individual prescription item.
- Four live data tabs: combined JOIN, Doctors, Consultations, Prescription Items.
- Validation for blank fields and invalid/negative medicine cost.
- Refresh all tables and delete selectors after every change.

**How to use:** Run the next code cell. On a computer with a graphical desktop, the Tkinter window will open. In cloud notebooks where desktop windows are unavailable, the database setup and helper functions will still run; use the notebook on your local computer to open the GUI.

## 1. Import libraries and configure the database

`sqlite3` is included with Python. `tkinter` is Python's standard GUI library. No extra package installation is normally required.

In [ ]:
import sqlite3
from pathlib import Path
from datetime import date
import tkinter as tk
from tkinter import ttk, messagebox

# Database file will be created in the same folder as this notebook.
DB_FILE = Path("consultation_dashboard.db")


def get_connection():
    """Open a database connection and enable foreign-key rules."""
    conn = sqlite3.connect(DB_FILE)
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def initialize_database():
    """Create tables and add sample records only when the database is empty."""
    with get_connection() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS doctors (
                doctor_id INTEGER PRIMARY KEY AUTOINCREMENT,
                doctor_name TEXT NOT NULL UNIQUE,
                email TEXT NOT NULL
            )
        """)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS consultations (
                consultation_id INTEGER PRIMARY KEY AUTOINCREMENT,
                doctor_id INTEGER NOT NULL,
                patient_name TEXT NOT NULL,
                consultation_date TEXT NOT NULL,
                FOREIGN KEY (doctor_id)
                    REFERENCES doctors(doctor_id)
                    ON DELETE CASCADE
            )
        """)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS prescription_items (
                prescription_id INTEGER PRIMARY KEY AUTOINCREMENT,
                consultation_id INTEGER NOT NULL,
                medicine_name TEXT NOT NULL,
                medicine_cost REAL NOT NULL CHECK (medicine_cost >= 0),
                FOREIGN KEY (consultation_id)
                    REFERENCES consultations(consultation_id)
                    ON DELETE CASCADE
            )
        """)

        # Seed sample data only if all three tables are empty.
        total = conn.execute("""
            SELECT
                (SELECT COUNT(*) FROM doctors) +
                (SELECT COUNT(*) FROM consultations) +
                (SELECT COUNT(*) FROM prescription_items)
        """).fetchone()[0]

        if total == 0:
            sample_doctors = [
                ("Dr. Ananya Rao", "ananya.rao@example.com"),
                ("Dr. Rahul Mehta", "rahul.mehta@example.com"),
                ("Dr. Priya Nair", "priya.nair@example.com")
            ]
            conn.executemany(
                "INSERT INTO doctors (doctor_name, email) VALUES (?, ?)",
                sample_doctors
            )

            doctor_ids = {
                name: doctor_id
                for doctor_id, name in conn.execute(
                    "SELECT doctor_id, doctor_name FROM doctors"
                ).fetchall()
            }

            sample_consultations = [
                (doctor_ids["Dr. Ananya Rao"], "Meera Nair", "2026-11-01"),
                (doctor_ids["Dr. Rahul Mehta"], "Sandeep Gupta", "2026-11-02"),
                (doctor_ids["Dr. Priya Nair"], "Kavya Sharma", "2026-11-03")
            ]
            conn.executemany("""
                INSERT INTO consultations
                    (doctor_id, patient_name, consultation_date)
                VALUES (?, ?, ?)
            """, sample_consultations)

            consultation_ids = [
                row[0] for row in conn.execute(
                    "SELECT consultation_id FROM consultations ORDER BY consultation_id"
                ).fetchall()
            ]

            sample_prescriptions = [
                (consultation_ids[0], "Paracetamol 500mg", 25.00),
                (consultation_ids[1], "Amoxicillin 250mg", 85.50),
                (consultation_ids[2], "Cetirizine 10mg", 18.00)
            ]
            conn.executemany("""
                INSERT INTO prescription_items
                    (consultation_id, medicine_name, medicine_cost)
                VALUES (?, ?, ?)
            """, sample_prescriptions)


def add_consultation_record(doctor_name, patient_name, medicine_name, medicine_cost,
                            email="not-provided@example.com"):
    """Create/reuse doctor, then add one consultation and one prescription."""
    doctor_name = doctor_name.strip()
    patient_name = patient_name.strip()
    medicine_name = medicine_name.strip()

    if not doctor_name or not patient_name or not medicine_name:
        raise ValueError("Doctor name, patient name, and medicine description are required.")

    try:
        cost = float(medicine_cost)
    except (TypeError, ValueError):
        raise ValueError("Medicine cost must be a number.")

    if cost < 0:
        raise ValueError("Medicine cost cannot be negative.")

    with get_connection() as conn:
        doctor = conn.execute(
            "SELECT doctor_id FROM doctors WHERE doctor_name = ?",
            (doctor_name,)
        ).fetchone()

        if doctor:
            doctor_id = doctor[0]
        else:
            cursor = conn.execute(
                "INSERT INTO doctors (doctor_name, email) VALUES (?, ?)",
                (doctor_name, email.strip() or "not-provided@example.com")
            )
            doctor_id = cursor.lastrowid

        cursor = conn.execute("""
            INSERT INTO consultations (doctor_id, patient_name, consultation_date)
            VALUES (?, ?, ?)
        """, (doctor_id, patient_name, date.today().isoformat()))
        consultation_id = cursor.lastrowid

        conn.execute("""
            INSERT INTO prescription_items
                (consultation_id, medicine_name, medicine_cost)
            VALUES (?, ?, ?)
        """, (consultation_id, medicine_name, cost))

    return consultation_id


def fetch_join_view():
    """Return the combined consultation and prescription view."""
    with get_connection() as conn:
        return conn.execute("""
            SELECT c.consultation_id, d.doctor_name, c.patient_name,
                   c.consultation_date, p.medicine_name, p.medicine_cost
            FROM consultations AS c
            JOIN doctors AS d ON d.doctor_id = c.doctor_id
            JOIN prescription_items AS p
                ON p.consultation_id = c.consultation_id
            ORDER BY c.consultation_id
        """).fetchall()


def delete_record(target_type, record_id):
    """Delete one selected record. Foreign keys perform required cascades."""
    table_and_key = {
        "Doctor": ("doctors", "doctor_id"),
        "Consultation": ("consultations", "consultation_id"),
        "Prescription Item": ("prescription_items", "prescription_id")
    }
    if target_type not in table_and_key:
        raise ValueError("Choose a valid record type.")

    table, key = table_and_key[target_type]
    with get_connection() as conn:
        cursor = conn.execute(
            f"DELETE FROM {table} WHERE {key} = ?",
            (int(record_id),)
        )
        if cursor.rowcount == 0:
            raise ValueError("The selected record no longer exists.")


## 2. Create the Tkinter dashboard

The dashboard uses `ttk.Notebook` for the four tabs and `ttk.Treeview` to display database records. The delete dropdown is rebuilt after every insert or delete so it cannot keep a deleted ID.

In [ ]:
class ConsultationDashboard:
    def __init__(self, root):
        self.root = root
        self.root.title("Online Consultation & Prescription Management")
        self.root.geometry("1180x760")
        self.root.minsize(900, 600)

        title = ttk.Label(
            root,
            text="ONLINE CONSULTATION & PRESCRIPTION MANAGEMENT",
            font=("Arial", 16, "bold")
        )
        title.pack(pady=(12, 3))

        subtitle = ttk.Label(
            root,
            text="Doctors  |  Consultations  |  Prescription Items",
            font=("Arial", 10)
        )
        subtitle.pack(pady=(0, 10))

        self.build_insert_panel()
        self.build_delete_panel()
        self.build_tabs()
        self.refresh_all()

    def build_insert_panel(self):
        panel = ttk.LabelFrame(self.root, text="Add New Consultation and Prescription")
        panel.pack(fill="x", padx=12, pady=6)

        self.doctor_var = tk.StringVar()
        self.email_var = tk.StringVar()
        self.patient_var = tk.StringVar()
        self.medicine_var = tk.StringVar()
        self.cost_var = tk.StringVar()

        fields = [
            ("Doctor Name *", self.doctor_var),
            ("Doctor Email (optional)", self.email_var),
            ("Patient Name *", self.patient_var),
            ("Medicine Description *", self.medicine_var),
            ("Medicine Cost *", self.cost_var)
        ]

        for index, (label_text, variable) in enumerate(fields):
            row = index // 3
            col = (index % 3) * 2
            ttk.Label(panel, text=label_text).grid(
                row=row, column=col, padx=8, pady=7, sticky="w"
            )
            ttk.Entry(panel, textvariable=variable, width=27).grid(
                row=row, column=col + 1, padx=8, pady=7, sticky="ew"
            )

        for col in range(6):
            panel.columnconfigure(col, weight=1)

        ttk.Button(
            panel, text="Add Record", command=self.add_record
        ).grid(row=1, column=5, padx=8, pady=7, sticky="e")

    def build_delete_panel(self):
        panel = ttk.LabelFrame(self.root, text="Delete Records")
        panel.pack(fill="x", padx=12, pady=6)

        self.delete_type_var = tk.StringVar(value="Doctor")
        self.delete_item_var = tk.StringVar()

        ttk.Label(panel, text="Record Type:").grid(
            row=0, column=0, padx=8, pady=8, sticky="w"
        )
        type_box = ttk.Combobox(
            panel,
            textvariable=self.delete_type_var,
            values=["Doctor", "Consultation", "Prescription Item"],
            state="readonly",
            width=22
        )
        type_box.grid(row=0, column=1, padx=8, pady=8, sticky="w")
        type_box.bind("<<ComboboxSelected>>", lambda event: self.refresh_delete_options())

        ttk.Label(panel, text="Select Record:").grid(
            row=0, column=2, padx=8, pady=8, sticky="w"
        )
        self.delete_combo = ttk.Combobox(
            panel, textvariable=self.delete_item_var,
            state="readonly", width=65
        )
        self.delete_combo.grid(row=0, column=3, padx=8, pady=8, sticky="ew")
        panel.columnconfigure(3, weight=1)

        ttk.Button(
            panel, text="Delete Selected", command=self.delete_selected
        ).grid(row=0, column=4, padx=8, pady=8)

    def build_tabs(self):
        self.tabs = ttk.Notebook(self.root)
        self.tabs.pack(fill="both", expand=True, padx=12, pady=8)

        self.tree_specs = {
            "Combined JOIN View": [
                ("Consultation ID", 110), ("Doctor Name", 160),
                ("Patient Name", 150), ("Consultation Date", 125),
                ("Medicine Name", 190), ("Medicine Cost", 110)
            ],
            "Doctors": [
                ("Doctor ID", 100), ("Doctor Name", 220), ("Email", 250)
            ],
            "Consultations": [
                ("Consultation ID", 130), ("Doctor ID", 100),
                ("Patient Name", 220), ("Consultation Date", 150)
            ],
            "Prescription Items": [
                ("Prescription ID", 130), ("Consultation ID", 140),
                ("Medicine Name", 240), ("Medicine Cost", 130)
            ]
        }

        self.trees = {}
        for tab_name, columns in self.tree_specs.items():
            frame = ttk.Frame(self.tabs)
            self.tabs.add(frame, text=tab_name)

            column_ids = [f"col_{i}" for i in range(len(columns))]
            tree = ttk.Treeview(
                frame, columns=column_ids, show="headings", height=14
            )
            for column_id, (heading, width) in zip(column_ids, columns):
                tree.heading(column_id, text=heading)
                tree.column(column_id, width=width, minwidth=80, anchor="center")

            vertical_scroll = ttk.Scrollbar(
                frame, orient="vertical", command=tree.yview
            )
            horizontal_scroll = ttk.Scrollbar(
                frame, orient="horizontal", command=tree.xview
            )
            tree.configure(
                yscrollcommand=vertical_scroll.set,
                xscrollcommand=horizontal_scroll.set
            )
            tree.grid(row=0, column=0, sticky="nsew")
            vertical_scroll.grid(row=0, column=1, sticky="ns")
            horizontal_scroll.grid(row=1, column=0, sticky="ew")
            frame.rowconfigure(0, weight=1)
            frame.columnconfigure(0, weight=1)
            self.trees[tab_name] = tree

    def add_record(self):
        try:
            if not self.doctor_var.get().strip():
                raise ValueError("Please enter the doctor name.")
            if not self.patient_var.get().strip():
                raise ValueError("Please enter the patient name.")
            if not self.medicine_var.get().strip():
                raise ValueError("Please enter the medicine description.")
            if not self.cost_var.get().strip():
                raise ValueError("Please enter the medicine cost.")

            cost = float(self.cost_var.get().strip())
            if cost < 0:
                raise ValueError("Medicine cost cannot be negative.")

            consultation_id = add_consultation_record(
                self.doctor_var.get(),
                self.patient_var.get(),
                self.medicine_var.get(),
                cost,
                self.email_var.get()
            )

            for variable in (
                self.doctor_var, self.email_var, self.patient_var,
                self.medicine_var, self.cost_var
            ):
                variable.set("")

            self.refresh_all()
            messagebox.showinfo(
                "Success",
                f"Consultation and prescription added successfully.\n"
                f"Consultation ID: {consultation_id}"
            )
        except ValueError as error:
            messagebox.showerror("Input Error", str(error))
        except sqlite3.IntegrityError as error:
            messagebox.showerror("Database Error", str(error))
        except Exception as error:
            messagebox.showerror("Unexpected Error", str(error))

    def delete_selected(self):
        selected = self.delete_combo.current()
        if selected < 0:
            messagebox.showwarning("Select Record", "Please select a record to delete.")
            return

        selected_value = self.delete_combo["values"][selected]
        record_id = selected_value.split(" | ", 1)[0]

        if not messagebox.askyesno(
            "Confirm Delete",
            f"Delete this {self.delete_type_var.get()} record?\n{selected_value}"
        ):
            return

        try:
            delete_record(self.delete_type_var.get(), record_id)
            self.refresh_all()
            messagebox.showinfo("Success", "Selected record deleted.")
        except Exception as error:
            messagebox.showerror("Delete Error", str(error))

    def fill_tree(self, tree, rows):
        for item in tree.get_children():
            tree.delete(item)
        for row in rows:
            formatted = tuple(
                f"₹{value:.2f}" if isinstance(value, float) else value
                for value in row
            )
            tree.insert("", "end", values=formatted)

    def refresh_all(self):
        with get_connection() as conn:
            doctors = conn.execute("""
                SELECT doctor_id, doctor_name, email
                FROM doctors ORDER BY doctor_id
            """).fetchall()

            consultations = conn.execute("""
                SELECT consultation_id, doctor_id, patient_name, consultation_date
                FROM consultations ORDER BY consultation_id
            """).fetchall()

            prescriptions = conn.execute("""
                SELECT prescription_id, consultation_id, medicine_name, medicine_cost
                FROM prescription_items ORDER BY prescription_id
            """).fetchall()

        self.fill_tree(self.trees["Combined JOIN View"], fetch_join_view())
        self.fill_tree(self.trees["Doctors"], doctors)
        self.fill_tree(self.trees["Consultations"], consultations)
        self.fill_tree(self.trees["Prescription Items"], prescriptions)
        self.refresh_delete_options()

    def refresh_delete_options(self):
        record_type = self.delete_type_var.get()
        with get_connection() as conn:
            if record_type == "Doctor":
                rows = conn.execute("""
                    SELECT doctor_id, doctor_name FROM doctors ORDER BY doctor_id
                """).fetchall()
                options = [f"{row[0]} | {row[1]}" for row in rows]
            elif record_type == "Consultation":
                rows = conn.execute("""
                    SELECT consultation_id, patient_name
                    FROM consultations ORDER BY consultation_id
                """).fetchall()
                options = [f"{row[0]} | Patient: {row[1]}" for row in rows]
            else:
                rows = conn.execute("""
                    SELECT prescription_id, medicine_name
                    FROM prescription_items ORDER BY prescription_id
                """).fetchall()
                options = [f"{row[0]} | {row[1]}" for row in rows]

        self.delete_combo["values"] = options
        self.delete_item_var.set(options[0] if options else "")


def launch_dashboard():
    """Launch the GUI where a desktop display is available."""
    try:
        root = tk.Tk()
    except tk.TclError as error:
        print("The database is ready, but this environment cannot open desktop Tkinter windows.")
        print("Run this notebook on your local computer with Python/Tkinter to use the GUI.")
        print("Display details:", error)
        return None

    ConsultationDashboard(root)
    root.mainloop()
    return root


# Prepare the database, then open the application.
initialize_database()
print("Database initialized at:", DB_FILE.resolve())
print("Doctors:", len(sqlite3.connect(DB_FILE).execute("SELECT * FROM doctors").fetchall()))
print("Consultations:", len(sqlite3.connect(DB_FILE).execute("SELECT * FROM consultations").fetchall()))
print("Prescription items:", len(sqlite3.connect(DB_FILE).execute("SELECT * FROM prescription_items").fetchall()))

launch_dashboard()


## 3. Database relationship and cascade-delete test

Run this optional cell to inspect the relationships and confirm that SQLite foreign-key enforcement is enabled. It does not delete any of your records.

In [ ]:
with get_connection() as conn:
    print("Foreign-key enforcement:", conn.execute("PRAGMA foreign_keys").fetchone()[0])
    print("\nDoctors:")
    for row in conn.execute("SELECT * FROM doctors"):
        print(row)

    print("\nCombined JOIN view:")
    for row in fetch_join_view():
        print(row)

    print("\nForeign-key relationships:")
    print("consultations -> doctors:",
          conn.execute("PRAGMA foreign_key_list(consultations)").fetchall())
    print("prescription_items -> consultations:",
          conn.execute("PRAGMA foreign_key_list(prescription_items)").fetchall())


## 4. How the project works

1. `get_connection()` opens the SQLite database and enables foreign keys for each connection.
2. `initialize_database()` creates the three tables and inserts sample data only when the database has no records.
3. `add_consultation_record()` checks whether the doctor already exists. It reuses that doctor's ID or creates a new doctor, then inserts the consultation and prescription.
4. `fetch_join_view()` combines all three tables using SQL `JOIN`.
5. `delete_record()` deletes the selected row. SQLite automatically deletes dependent child rows because of `ON DELETE CASCADE`.
6. `refresh_all()` reloads all four tabs and rebuilds the dropdown after an operation.

### Important database rules
- Deleting a **Doctor** also deletes that doctor's consultations and their prescription items.
- Deleting a **Consultation** also deletes its prescription items.
- Deleting a **Prescription Item** deletes only that prescription row. The consultation and doctor remain.

### Notes
- The sample doctor emails and patient records are fictional demonstration data.
- The optional email field is used only when a new doctor is created. If an existing doctor is entered, their stored profile is reused.
- The database file is named `consultation_dashboard.db` and is created in the notebook's current working folder.
- For a fresh demonstration, close the app and delete `consultation_dashboard.db`; running the setup again will recreate it with sample data.
- If Tkinter cannot open in a cloud notebook, the database functions still work there, but the graphical interface requires a local desktop Python environment.